# 未知データ（-12℃）での検証

学習・既存の検証（-6℃）には一切使用していない -12℃ データを使って、
① YOLO単体（検出精度）、② ResNet単体（粒径予測精度）、③ パイプライン全体
の3つをそれぞれ検証する。

## セットアップ

In [ ]:
%matplotlib inline
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import matplotlib.pyplot as plt

for _font in ["Yu Gothic", "Meiryo", "MS Gothic"]:
    if _font in {f.name for f in plt.matplotlib.font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

from resnet_utils import build_model, make_resnet_transform

IMAGE_ROOT = r"g:\マイドライブ\【野中用】氷結晶画像BOX\-12℃"
CSV_ROOT = r"g:\マイドライブ\【野中用】氷結晶画像BOX\Excelデータ\-12℃"
YOLO_WEIGHTS = "../yolo_project/runs/detect/crystal_yolo-10/weights/best.pt"
RESNET_WEIGHTS = "../vscode_project/outputs/best_model.pth"

PATCH_SIZE = 640
OVERLAP_RATIO = 0.15
CONF = 0.25

COL_DIAMETER = "円相当径"
COL_X1 = "矩形領域(左上:x)"
COL_Y1 = "矩形領域(左上:y)"
COL_X2 = "矩形領域(右下:x)"
COL_Y2 = "矩形領域(右下:y)"
IMG_EXTS = (".bmp", ".tif", ".tiff", ".jpg", ".png")
EXCLUDE_MINUTES = {"0"}
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}  # 学習時(-6℃)の校正値
BASE_UM_PER_PIXEL = MAG_TO_UM_PER_PIXEL[40]  # 学習時の基準スケール(40倍)
NEW_MAG_TO_UM_PER_PIXEL = {40: 0.056119, 20: 0.10984, 10: 0.221}  # -12℃カメラの実際の校正値
MIN_GT_DIAMETER_UM = 10 * BASE_UM_PER_PIXEL  # 旧基準で10px相当の物理サイズを、ノイズ除外の閾値として固定
MIN_GT_DIAMETER_PX = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")

assert Path(IMAGE_ROOT).exists(), f"{IMAGE_ROOT} が見つかりません。IMAGE_ROOTを実際のパスに変更してください。"
assert Path(CSV_ROOT).exists(), f"{CSV_ROOT} が見つかりません。CSV_ROOTを実際のパスに変更してください。"
assert Path(YOLO_WEIGHTS).exists(), f"{YOLO_WEIGHTS} が見つかりません。"
assert Path(RESNET_WEIGHTS).exists(), f"{RESNET_WEIGHTS} が見つかりません。"

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=YOLO_WEIGHTS,
    confidence_threshold=CONF,
    device=str(device),
)

resnet = build_model().to(device)
resnet.load_state_dict(torch.load(RESNET_WEIGHTS, map_location=device))
resnet.eval()
transform = make_resnet_transform()

print("準備完了")

## 共通関数・画像/CSVペアの取得

In [ ]:
def normalize_folder_name(name):
    return re.sub(r"[（）()・\s]", "", name)

def get_minute_mag_key(filename):
    m = re.search(r"(\d+)分.*?(\d+)倍", filename)
    return m.groups() if m else None

def find_image_csv_pairs(image_root, csv_root):
    img_dirs = {normalize_folder_name(d): d
                for d in os.listdir(image_root)
                if os.path.isdir(os.path.join(image_root, d))}
    csv_dirs = {normalize_folder_name(d): d
                for d in os.listdir(csv_root)
                if os.path.isdir(os.path.join(csv_root, d))}
    common = set(img_dirs) & set(csv_dirs)
    pairs = []
    for key in sorted(common):
        image_dir = os.path.join(image_root, img_dirs[key])
        csv_dir = os.path.join(csv_root, csv_dirs[key])
        images = defaultdict(list)
        for f in os.listdir(image_dir):
            if f.lower().endswith(IMG_EXTS):
                k = get_minute_mag_key(f)
                if k:
                    images[k].append(f)
        csvs = defaultdict(list)
        for f in os.listdir(csv_dir):
            if f.lower().endswith(".csv"):
                k = get_minute_mag_key(f)
                if k:
                    csvs[k].append(f)
        for k in sorted(set(images) & set(csvs)):
            minute, mag = k
            if minute in EXCLUDE_MINUTES:
                continue
            mag = int(mag)
            if mag not in MAG_TO_UM_PER_PIXEL:
                continue
            for img_f, csv_f in zip(sorted(images[k]), sorted(csvs[k])):
                pairs.append((os.path.join(image_dir, img_f), os.path.join(csv_dir, csv_f), mag))
    print(f"画像・CSVペア: {len(pairs)}件")
    return pairs

pairs = find_image_csv_pairs(IMAGE_ROOT, CSV_ROOT)


def get_pred_boxes_and_diameters(image_path, magnification, conf=0.25):
    # -12℃データは撮影カメラが違い、(1)色味(茶色っぽさ)、(2)ピクセルスケール(µm/px)が
    # YOLOの学習時と異なる。検証の結果、グレースケール化＋スケール補正の縮小を両方かけると
    # マッチ率(gt側)が50.8%→72.0%まで改善したため、検出時はこの前処理を使う。
    # ResNet用のcropは元のRGB・元解像度の画像から取るので、この前処理の影響を受けない
    # （ResNet側はmake_resnet_transform()で自前にグレースケール化しているため元々頑健）。
    RESIZE_FACTOR = {40: 0.056119/0.088725, 20: 0.10984/0.17353, 10: 0.221/0.34392}
    detection_model.confidence_threshold = conf
    orig_img = Image.open(image_path).convert("RGB")
    gray_img = orig_img.convert("L").convert("RGB")
    factor = RESIZE_FACTOR.get(magnification, 1.0)
    det_img = gray_img.resize((round(gray_img.width * factor), round(gray_img.height * factor))) if factor != 1.0 else gray_img
    sliced_result = get_sliced_prediction(
        np.array(det_img), detection_model,
        slice_height=PATCH_SIZE, slice_width=PATCH_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    W, H = orig_img.size
    crop_scale = NEW_MAG_TO_UM_PER_PIXEL[magnification] / BASE_UM_PER_PIXEL
    boxes, diameters = [], []
    with torch.no_grad():
        for pred in sliced_result.object_prediction_list:
            # 縮小した画像上の座標を、元の画像サイズの座標に戻す
            x1 = max(0, int(pred.bbox.minx / factor)); y1 = max(0, int(pred.bbox.miny / factor))
            x2 = min(W, int(pred.bbox.maxx / factor)); y2 = min(H, int(pred.bbox.maxy / factor))
            if x2 <= x1 or y2 <= y1:
                continue
            crop = orig_img.crop((x1, y1, x2, y2))
            if crop_scale != 1.0:
                new_w = max(1, round(crop.width * crop_scale))
                new_h = max(1, round(crop.height * crop_scale))
                crop = crop.resize((new_w, new_h))
            x = transform(crop).unsqueeze(0).to(device)
            diameters.append(resnet(x).item())
            boxes.append((x1, y1, x2, y2))
    return boxes, diameters


def get_gt_boxes_diameters_circularity(csv_path, magnification):
    df = pd.read_csv(csv_path, encoding="cp932")
    diameter_col = next((c for c in df.columns if COL_DIAMETER in str(c)), None)
    circ_col = next((c for c in df.columns if "円形度" in str(c)), None)
    boxes, diameters_um, circularities = [], [], []
    for _, row in df.iterrows():
        try:
            x1, y1 = int(row[COL_X1]), int(row[COL_Y1])
            x2, y2 = int(row[COL_X2]), int(row[COL_Y2])
            d_px = float(row[diameter_col])
        except (KeyError, ValueError, TypeError):
            continue
        d_um = d_px * NEW_MAG_TO_UM_PER_PIXEL[magnification]
        if d_um < MIN_GT_DIAMETER_UM:
            continue
        left, right = sorted([x1, x2]); top, bottom = sorted([y1, y2])
        if right <= left or bottom <= top:
            continue
        circ = None
        if circ_col is not None:
            try:
                circ = float(row[circ_col])
            except (ValueError, TypeError):
                circ = None
        boxes.append((left, top, right, bottom))
        diameters_um.append(d_um)
        circularities.append(circ)
    return boxes, diameters_um, circularities


def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def match_boxes(pred_boxes, gt_boxes, iou_threshold=0.3):
    used_gt = set()
    matches = []
    for pi, pbox in enumerate(pred_boxes):
        best_j, best_iou = -1, iou_threshold
        for gj, gbox in enumerate(gt_boxes):
            if gj in used_gt:
                continue
            v = iou(pbox, gbox)
            if v > best_iou:
                best_iou, best_j = v, gj
        if best_j >= 0:
            matches.append((pi, best_j, best_iou))
            used_gt.add(best_j)
    return matches

print("関数定義完了")

## ① YOLO単体の検証（検出・マッチング精度）

-12℃の画像に対してYOLOで検出し、手作業の結晶とIoU≥0.3でマッチングできた割合を見る。
（-6℃の時に作ったVenn図と同じ考え方）

In [ ]:
total_pred, total_gt, total_matched = 0, 0, 0
per_image_rows = []
for i, (image_path, csv_path, mag) in enumerate(pairs):
    try:
        pred_boxes, pred_diameters = get_pred_boxes_and_diameters(image_path, mag)
        gt_boxes, gt_diameters, gt_circularities = get_gt_boxes_diameters_circularity(csv_path, mag)
    except Exception:
        continue
    matches = match_boxes(pred_boxes, gt_boxes, iou_threshold=0.3)
    n_pred, n_gt, n_matched = len(pred_boxes), len(gt_boxes), len(matches)
    total_pred += n_pred
    total_gt += n_gt
    total_matched += n_matched
    per_image_rows.append({
        "image_path": str(image_path),
        "n_pred": n_pred, "n_gt": n_gt, "n_matched": n_matched,
        "match_rate_of_pred": n_matched / n_pred if n_pred else float("nan"),
        "match_rate_of_gt": n_matched / n_gt if n_gt else float("nan"),
    })
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(pairs)} 画像処理済み")

match_rate_df = pd.DataFrame(per_image_rows)
print(f"\n=== -12℃（未知データ）でのYOLO検出 ===")
print(f"YOLO検出の総数: {total_pred}")
print(f"手作業結晶の総数: {total_gt}")
print(f"マッチできたペアの総数: {total_matched}")
if total_pred:
    print(f"YOLO検出のうちマッチできた割合: {total_matched/total_pred*100:.1f}%")
if total_gt:
    print(f"手作業結晶のうちマッチできた割合: {total_matched/total_gt*100:.1f}%")
print()
print(match_rate_df[["n_pred", "n_gt", "n_matched", "match_rate_of_pred", "match_rate_of_gt"]].describe())

## ② ResNet単体の検証（正しい位置のcropを与えた場合の粒径予測精度）

YOLOを介さず、手作業の正解boxから直接cropしてResNetに入力する。
ResNet自体が未知データの結晶サイズをどれだけ正確に読み取れるかを見る。

In [ ]:
def predict_diameter_from_box(orig_img, box, magnification):
    x1, y1, x2, y2 = box
    crop = orig_img.crop((x1, y1, x2, y2))
    crop_scale = NEW_MAG_TO_UM_PER_PIXEL[magnification] / BASE_UM_PER_PIXEL
    if crop_scale != 1.0:
        new_w = max(1, round(crop.width * crop_scale))
        new_h = max(1, round(crop.height * crop_scale))
        crop = crop.resize((new_w, new_h))
    x = transform(crop).unsqueeze(0).to(device)
    with torch.no_grad():
        return resnet(x).item()


resnet_records = []
for i, (image_path, csv_path, mag) in enumerate(pairs):
    try:
        gt_boxes, gt_diameters, gt_circularities = get_gt_boxes_diameters_circularity(csv_path, mag)
    except Exception:
        continue
    orig_img = Image.open(image_path).convert("RGB")  # 画像は1枚につき1回だけ開く（結晶ごとに開き直さない）
    for box, manual_um in zip(gt_boxes, gt_diameters):
        pred_um = predict_diameter_from_box(orig_img, box, mag)
        resnet_records.append({
            "image": os.path.basename(image_path),
            "magnification": mag,
            "manual_um": manual_um,
            "pred_um": pred_um,
        })
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(pairs)} 画像処理済み")

resnet_df = pd.DataFrame(resnet_records)
print(f"\nResNet単体検証（-12℃）: {len(resnet_df)}件")

In [ ]:
y_obs = resnet_df["manual_um"].to_numpy()
y_pred = resnet_df["pred_um"].to_numpy()

yvalues = np.concatenate([y_obs, y_pred])
ymin, ymax, yrange = yvalues.min(), yvalues.max(), np.ptp(yvalues)

ss_res = ((y_pred - y_obs) ** 2).sum()
ss_tot = ((y_obs - y_obs.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

plt.figure(figsize=(7, 7))
plt.scatter(y_obs, y_pred, alpha=0.3, s=10)
plt.plot([ymin - yrange*0.01, ymax + yrange*0.01], [ymin - yrange*0.01, ymax + yrange*0.01], "r--")
plt.xlim(ymin - yrange*0.01, ymax + yrange*0.01)
plt.ylim(ymin - yrange*0.01, ymax + yrange*0.01)
plt.xlabel("手作業 粒径 (µm)")
plt.ylabel("ResNet予測 粒径 (µm)")
plt.title(f"ResNet単体 yyplot（-12℃未知データ, n={len(resnet_df)}, R²={r2:.4f})")
plt.show()

rmse = np.sqrt(((y_pred - y_obs) ** 2).mean())
mae = np.abs(y_pred - y_obs).mean()
print(f"RMSE: {rmse:.3f}  MAE: {mae:.3f}  R²: {r2:.4f}")

## ③ パイプライン全体の検証（YOLO→ResNet）

YOLOの検出結果をそのままResNetに通す、本番と同じ流れで検証する。

In [ ]:
records = []
for i, (image_path, csv_path, mag) in enumerate(pairs):
    try:
        pred_boxes, pred_diameters = get_pred_boxes_and_diameters(image_path, mag)
        gt_boxes, gt_diameters, gt_circularities = get_gt_boxes_diameters_circularity(csv_path, mag)
    except Exception:
        continue
    if not pred_boxes or not gt_boxes:
        continue
    matches = match_boxes(pred_boxes, gt_boxes, iou_threshold=0.3)
    for pi, gj, iou_val in matches:
        records.append({
            "image": os.path.basename(image_path),
            "magnification": mag,
            "pred_um": pred_diameters[pi],
            "manual_um": gt_diameters[gj],
        })
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(pairs)} 画像処理済み  マッチ累計: {len(records)}件")

match_df_unseen = pd.DataFrame(records)
match_df_unseen["diff_pct"] = (match_df_unseen["pred_um"] / match_df_unseen["manual_um"] - 1) * 100
print(f"\nマッチした結晶数（-12℃パイプライン全体）: {len(match_df_unseen)}件")

In [ ]:
y_obs = match_df_unseen["manual_um"].to_numpy()
y_pred = match_df_unseen["pred_um"].to_numpy()

yvalues = np.concatenate([y_obs, y_pred])
ymin, ymax, yrange = yvalues.min(), yvalues.max(), np.ptp(yvalues)

ss_res = ((y_pred - y_obs) ** 2).sum()
ss_tot = ((y_obs - y_obs.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

plt.figure(figsize=(7, 7))
plt.scatter(y_obs, y_pred, alpha=0.2, s=8)
plt.plot([ymin - yrange*0.01, ymax + yrange*0.01], [ymin - yrange*0.01, ymax + yrange*0.01], "r--")
plt.xlim(ymin - yrange*0.01, ymax + yrange*0.01)
plt.ylim(ymin - yrange*0.01, ymax + yrange*0.01)
plt.xlabel("手作業 粒径 (µm)")
plt.ylabel("自動パイプライン 粒径 (µm)")
plt.title(f"パイプライン全体 yyplot（-12℃未知データ, n={len(match_df_unseen)}, R²={r2:.4f})")
plt.show()

rmse = np.sqrt(((y_pred - y_obs) ** 2).mean())
mae = np.abs(y_pred - y_obs).mean()
print(f"RMSE: {rmse:.3f}  MAE: {mae:.3f}  R²: {r2:.4f}")

## ④ 再結晶化速度定数(k値)の検証

①②③と同じ補正済みの関数（グレースケール化＋スケール補正＋正しい校正値）を使い、
画像ごとの平均粒径からk値を算出して、手作業 vs 自動パイプラインで比較する。

In [ ]:
def get_minute(filename):
    m = re.search(r"(\d+)分", filename)
    return int(m.group(1)) if m else None

def get_sample(image_path):
    return Path(image_path).parent.name


records = []
for i, (image_path, csv_path, mag) in enumerate(pairs):
    minute = get_minute(os.path.basename(image_path))
    if minute is None:
        continue
    try:
        _, gt_diameters, _ = get_gt_boxes_diameters_circularity(csv_path, mag)
        _, pred_diameters = get_pred_boxes_and_diameters(image_path, mag)
    except Exception:
        continue
    if len(gt_diameters) == 0 or len(pred_diameters) == 0:
        continue
    gt_arr = np.array(gt_diameters)
    pred_arr = np.array(pred_diameters)
    records.append({
        "sample": get_sample(image_path),
        "magnification": mag,
        "minute": minute,
        "mean_manual_um": gt_arr.mean(),
        "mean_pred_um": pred_arr.mean(),
        "R3_manual": (gt_arr.mean() / 2) ** 3,
        "R3_pred": (pred_arr.mean() / 2) ** 3,
        "n_manual": len(gt_arr),
        "n_pred": len(pred_arr),
    })
    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(pairs)} 画像処理済み")

growth_df_unseen = pd.DataFrame(records)
print(f"\n集計できた画像: {len(growth_df_unseen)}件")
growth_df_unseen.sort_values(["sample", "minute"])

In [ ]:
def fit_k(df, r3_col):
    df = df.sort_values("minute")
    t = df["minute"].to_numpy(dtype=float)
    R3 = df[r3_col].to_numpy(dtype=float)
    if len(t) < 2:
        return np.nan, np.nan
    A = np.column_stack([np.ones(len(t)), t])
    coef, *_ = np.linalg.lstsq(A, R3, rcond=None)
    R0_3, k = coef
    return k, R0_3


per_time = growth_df_unseen.groupby(["sample", "minute"]).agg(
    R3_manual=("R3_manual", "mean"),
    R3_pred=("R3_pred", "mean"),
).reset_index()

results = []
for sample, g in per_time.groupby("sample"):
    if g["minute"].nunique() < 2:
        continue
    k_manual, _ = fit_k(g, "R3_manual")
    k_pred, _ = fit_k(g, "R3_pred")
    results.append({
        "sample": sample,
        "n_timepoints": g["minute"].nunique(),
        "k_manual": k_manual, "k_pred": k_pred,
    })

k_df_unseen = pd.DataFrame(results)
print(f"フィッティングできたサンプル: {len(k_df_unseen)}件\n")
print(k_df_unseen.to_string(index=False))

valid = k_df_unseen.dropna(subset=["k_manual", "k_pred"])
corr = valid["k_manual"].corr(valid["k_pred"])
ss_res = ((valid["k_pred"] - valid["k_manual"]) ** 2).sum()
ss_tot = ((valid["k_manual"] - valid["k_manual"].mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
rmse = np.sqrt(ss_res / len(valid))
mae = (valid["k_pred"] - valid["k_manual"]).abs().mean()

print(f"\n相関係数: {corr:.4f}")
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.3f}  MAE: {mae:.3f}")

plt.figure(figsize=(6, 6))
plt.scatter(valid["k_manual"], valid["k_pred"])
lims = [min(valid["k_manual"].min(), valid["k_pred"].min()) * 1.1,
        max(valid["k_manual"].max(), valid["k_pred"].max()) * 1.1]
plt.plot(lims, lims, "r--", linewidth=1)
plt.xlabel("手作業 k")
plt.ylabel("自動パイプライン k")
plt.title(f"再結晶化速度定数 k: 手作業 vs 自動（-12℃未知データ, R²={r2:.4f})")
plt.show()